# Mamba-CL on Split-CIFAR100 (20 tasks x 5 classes): De-focus Mamba **base** backbone

Mamba-CL protocol (10 epochs/task, null-space, seed 2024) with `defocus_mamba_base` (24 layers, 768-dim, ~92M params,
ImageNet-1K pretrained) instead of `defocus_mamba_large` (48 layers, 1024-dim, ~330M, ImageNet-21K pretrained).
Our large-backbone reference run was 10 tasks x 10 classes: AA 90.16%, FM 2.18%.

Environment matches the RunPod setup: Python 3.11 venv, torch 2.3.1 + cu121, causal-conv1d 1.4.0, mamba-ssm 1.2.2.

Settings: **GPU T4**, **Internet on**. W&B: add a Kaggle Secret named `WANDB_API_KEY` and attach it to this notebook
(Add-ons > Secrets). Without it the run logs to W&B offline; sync later with `wandb sync` on the saved `wandb/` folder.
Kaggle stops sessions at 12 h, so check the projected total printed after task 1.
This version is a **timing run** (1 epoch/task, batch 128) and prints the full log with progress bars every 30 s.

In [ ]:
# ---- run parameters ----
REPO     = "https://github.com/HasinthakaPiyumal/hippo-cortex.git"
BRANCH   = "feat/kaggle-progress-logging"
MODEL    = "defocus_mamba_base"
TASKS    = 20        # Split-CIFAR100 20x5 (5 classes per task)
EPOCHS   = 1         # timing run; Mamba-CL paper protocol is 10
BATCH    = 128       # 200 ran out of memory on T4 (large backbone, task 2)
SEED     = 2024
USE_AMP  = True
WORKERS  = 4
WANDB_PROJECT = "mamba-cl-baseline"
PROGRESS_INTERVAL = 30   # seconds between progress-bar updates in the log

In [ ]:
!nvidia-smi
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("hippo-cortex"):
    # Fall back to main once the branch is merged (GitHub deletes merged branches here).
    if subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO]).returncode != 0:
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
ROOT = "/kaggle/working/hippo-cortex"
VENV = "/kaggle/working/py311_env"
PY, PIP = f"{VENV}/bin/python", f"{VENV}/bin/pip"
!cd {ROOT} && git log --oneline -1

In [ ]:
# Python 3.11 venv (the Kaggle image's Python is too new for torch 2.3.1 / mamba-ssm 1.2.2)
!add-apt-repository -y ppa:deadsnakes/ppa > /dev/null
!apt-get update -y -qq > /dev/null
!apt-get install -y -qq python3.11 python3.11-venv python3.11-dev python3.11-distutils > /dev/null
!python3.11 -m venv {VENV}
!{PY} -m ensurepip
!{PIP} install -q --upgrade pip
!{PIP} install -q torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121
!{PIP} install -q ninja packaging wheel
!{PIP} install -q --no-build-isolation causal-conv1d==1.4.0
!{PIP} install -q --no-build-isolation mamba-ssm==1.2.2
!cd {ROOT} && {PIP} install -q -r requirements.txt && {PIP} install -q -U wandb && {PIP} install -q --no-deps -e .
!{PY} -c "import torch, mamba_ssm, causal_conv1d; from mamba_ssm.ops.selective_scan_interface import selective_scan_fn; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| mamba fast path OK')"

In [ ]:
CKPT = "/kaggle/working/defocus_mamba_base_cls.pth"
if not os.path.exists(CKPT):
    !wget -q -O {CKPT} https://github.com/OpenGVLab/De-focus-Attention-Networks/releases/download/v1.0/defocus_mamba_base_cls.pth
DATA = "/tmp/cifar100-images"
if not os.path.exists(f"{DATA}/val"):
    !cd {ROOT} && {PY} scripts/download_datasets.py --dataset cifar100 && {PY} scripts/split_cifar100.py && cp -r data/cifar100-images /tmp/
!ls -la {CKPT}; ls {DATA}/train | wc -l; ls {DATA}/val | wc -l

In [ ]:
# W&B key comes from a Kaggle Secret; it is never written into this notebook.
env = dict(os.environ, WANDB_PROJECT=WANDB_PROJECT, WANDB_DIR="/kaggle/working")
try:
    from kaggle_secrets import UserSecretsClient
    env["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    print("W&B: online (key from Kaggle Secret)")
except Exception as e:
    env["WANDB_MODE"] = "offline"
    print("W&B: secret WANDB_API_KEY not attached, logging offline:", type(e).__name__)

In [ ]:
import time
LOG = f"/kaggle/working/mamba_cl_{MODEL}_t{TASKS}_e{EPOCHS}_s{SEED}.log"
cmd = [PY, "-u", "train_eval.py", "-d", "cifar100", "-m", MODEL,
       "-t", str(TASKS), "-e", str(EPOCHS), "-b", str(BATCH),
       "--pretrained_path", CKPT, "--data_root", DATA,
       "--null_eta", "0.95", "--use_null_space", "--seed", str(SEED),
       "--use_amp", str(USE_AMP), "-jt", str(WORKERS), "--use_wandb",
       "--progress_interval", str(PROGRESS_INTERVAL)]
print(" ".join(cmd))
t0 = time.time(); done = 0
with open(LOG, "w") as f:
    p = subprocess.Popen(cmd, cwd=f"{ROOT}/baselines/mamba-cl", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        f.write(line); f.flush()
        print(line, end="", flush=True)   # full log: epochs, progress bars, null-space timing
        if "Results of task" in line:
            done += 1
            el = time.time() - t0
            print(f"   elapsed {el/3600:.2f} h, projected total {el/done*TASKS/3600:.2f} h", flush=True)
    p.wait()
print("exit code", p.returncode, f"in {(time.time()-t0)/3600:.2f} h; full log: {LOG}")

In [ ]:
!tail -n 40 {LOG}